# Self-RAG Experimentation

This notebook breaks down and tests the individual components that make up the **Self-RAG** workflow in our Agentic RAG pipeline. We'll experiment with:
1. **Hallucination Grader**: Checking if an answer is grounded in the retrieved documents.
2. **Answer Quality Grader**: Checking if a generated answer actually resolves the user's query.
3. **Query Rewriter**: Reformulating a failed query into a better one.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from langchain_groq import ChatGroq
from src.config import SELF_RAG_MODEL, GROQ_API_KEY, TEMPERATURE, MAX_TOKENS

llm = ChatGroq(
    model=SELF_RAG_MODEL,
    api_key=GROQ_API_KEY,
    temperature=0,
    max_tokens=MAX_TOKENS
)
print("LLM Initialized!")

c:\Users\amanm\Desktop\learning\developer-chat-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM Initialized!


## 1. Hallucination Grader
Ensures that the LLM didn't invent facts that aren't present in the provided context.

In [2]:
def hallucination_grader(context: str, answer: str):
    prompt = f"""
    You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts.
    Give a binary score 'yes' or 'no'. 'Yes' means that the answer is grounded in and supported by the set of facts.
    
    Set of facts: {context}
    
    LLM generation: {answer}
    
    Binary Score ("yes" or "no"):
    """
    response = llm.invoke(prompt)
    return response.content.strip().lower()

context = "The Eiffel Tower is located in Paris, France. It was constructed in 1889."

In [3]:
# Test 1: Grounded Answer
good_answer = "The Eiffel Tower was built in 1889 and is located in Paris."
print("Grounded Score:", hallucination_grader(context, good_answer))

Grounded Score: yes


In [4]:
# Test 2: Hallucinated Answer
bad_answer = "The Eiffel Tower is located in Paris, France, and was built by aliens in 1889."
print("Hallucination Score:", hallucination_grader(context, bad_answer))

Hallucination Score: no


## 2. Answer Quality Grader
Ensures the generated answer actually answers the user's specific question (preventing tangential or evasive answers).

In [5]:
def answer_quality_grader(question: str, answer: str):
    prompt = f"""
    You are a grader assessing whether an answer addresses / resolves a question.
    Give a binary score 'yes' or 'no'. Yes means that the answer resolves the question.
    
    Question: {question}
    
    Answer: {answer}
    
    Binary Score ("yes" or "no"):
    """
    response = llm.invoke(prompt)
    return response.content.strip().lower()

query = "How tall is the Eiffel Tower?"

In [6]:
# Test 1: Useful Answer
good_answer = "The Eiffel Tower is 330 meters tall."
print("Quality Score:", answer_quality_grader(query, good_answer))

Quality Score: yes


In [7]:
# Test 2: Evasive Answer
bad_answer = "The Eiffel Tower is located in Paris, France and was built in 1889."
print("Quality Score:", answer_quality_grader(query, bad_answer))

Quality Score: no


## 3. Query Rewriter
If the retrieval fails to find good context, or if the answer is completely hallucinated and we need a new angle, we rewrite the original query.

In [8]:
def rewrite_query(question: str):
    prompt = f"""
    You are a question re-writer that converts an input question to a better version that is optimized 
    for vectorstore retrieval. Look at the input and try to reason about the underlying semantic intent / meaning.
    
    Initial question: {question}
    
    Formulate an improved question:
    """
    response = llm.invoke(prompt)
    return response.content.strip()


In [9]:
ambiguous_query = "iphone battery fast drain fix"
print("Original:", ambiguous_query)
print("Rewritten:", rewrite_query(ambiguous_query))

Original: iphone battery fast drain fix
Rewritten: **Improved question:**  
How can I stop my iPhone battery from draining so quickly?
